HOG

In [3]:
import cv2
import numpy as np

# 1. INICIALIZAÇÃO DO MÉTODO
# Instancia o descritor HOG
hog = cv2.HOGDescriptor()

# Carrega o SVM linear já treinado
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# Abre o arquivo de vídeo 
cap = cv2.VideoCapture('videos/vid1.mp4')

# Verifica se o vídeo foi aberto corretamente
if not cap.isOpened():
    print("Erro ao abrir o vídeo. Verifique o caminho do arquivo.")
    exit()

while True:
    # Lê o frame atual do vídeo
    ret, frame = cap.read()
    
    # Se o vídeo acabar (ou houver erro de leitura), encerra o loop
    if not ret:
        break

    # 2. OTIMIZAÇÃO: Redimensionamento
    # Veículos autônomos precisam de processamento rápido. Reduzir a matriz da imagem ajuda.
    frame = cv2.resize(frame, (800, 600))

    # 3. ROI - Region of Interest
    # A câmera do carro filma o céu e os prédios, onde não há pedestres.
    # Recortamos a matriz para analisar apenas da metade para baixo (rua e calçada).
    # Aqui, pegamos da linha 300 até a 600 (eixo Y), e todas as colunas (eixo X).
    roi = frame[300:600, :]

    # 4. Pré-processamento Clássico
    # Converte a ROI de BGR (3 canais) para Tons de Cinza (1 canal).
    # O HOG precisa apenas da variação de intensidade (gradiente) para achar o formato.
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    
    # Filtro de Suavização (Gaussiano). 
    # Remove ruídos.
    gray_blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # 5. DETECÇÃO (Janela Deslizante e Pirâmide de Escala)
    # winStride: O salto da janela ao varrer a imagem (8x8 pixels).
    # scale: O fator de redução da imagem na pirâmide (1.05 = reduz 5% a cada nível).
    # Retorna as 'boxes' (coordenadas dos retângulos) onde o SVM classificou como pedestre.
    boxes, weights = hog.detectMultiScale(gray_blur, winStride=(8, 8), padding=(8, 8), scale=1.05)

    # 6. PÓS-PROCESSAMENTO: Mapeamento de Coordenadas e Exibição

    # O 'weights' é um array 2D (Nx1). Convertendo para 1D 
    pesos = np.ravel(weights)
    
    for i, (x, y, w, h) in enumerate(boxes):
        # IMPORTANTE: A detecção foi feita na 'roi' (que começou no pixel Y = 300).
        # Para desenhar no 'frame' original colorido, precisamos somar esse valor ao eixo Y.
        y_real = y + 300
        
        confianca = pesos[i]

        # Desenha o retângulo verde ao redor do pedestre
        cv2.rectangle(frame, (x, y_real), (x + w, y_real + h), (0, 255, 0), 2)
        
        texto = f'Pedestre: {confianca:.2f}'

        # Adiciona um texto de identificação
        cv2.putText(frame, texto, (x, y_real - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow('Visao do Veiculo Autonomo - PDI', frame)


    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()